# LinkedIn Learning: Data Quality Upstream Project Walkthrough

## Setup and Background

### Github Codespaces

> [!NOTE]  
> I highly suggest starting GitHub Codespaces right away and let it build in the background while discuss the project and mock scenario. Note that the sandbox environment will take a few minutes as it's building an entire data platform and running it end-to-end to ensure it works properly.

Starting up codespaces is simple:
1. Click the green "< > Code" button.
2. Select "Codespaces."
3. Press the "+" button.
4. Wait for the environment to build for a couple minutes.
5. Start coding with a real-world environment in the browser!

<img src="course_assets/project_walkthrough_images/00_01_start_codespaces.png" width="1000">

### Scenario:

In this project you are a data engineer that's been tasked with exploring data quality enhancements to the company's new data platform. Given that the company is relatively early in their data maturity, you want to first understand what could potentially break the platform and provide a product requirements document (PRD) for your tech lead's review.

To support your work, an isolated sandbox environment has been created with the data platform and a sample of the data. By the end of the project, you should have a PRD with the following:
- An overview of the data platform architecture.
- Potential gaps in the data platform for the following areas:
    1. Ingestion into the transactional database.
    2. Transactions on the transactional database.
    3. Replication from the transactional database to the data lakehouse.
- Suggestions on improvements, their respective tradeoffs, and results of your implementation tests.
- Remaining questions that should be addressed but are outside the scope of this project.

### Data:

This project utilizes two public government datasets sourced from NYC Open Data:

**1. [NYC Parking Violations Issued - Fiscal Year 2023](https://data.cityofnewyork.us/City-Government/Parking-Violations-Issued-Fiscal-Year-2023/pvqr-7yc4)**  
> Parking Violations Issuance datasets contain violations issued during the respective fiscal year. The Issuance datasets are not updated to reflect violation status, the information only represents the violation(s) at the time they are issued. Since appearing on an issuance dataset, a violation may have been paid, dismissed via a hearing, statutorily expired, or had other changes to its status. To see the current status of outstanding parking violations, please look at the Open Parking & Camera Violations dataset.

Note that this dataset is rather large for this project, so we will use a sample of 100K records.

**2. [NYC Department of Finance Parking Violation Codes](https://data.cityofnewyork.us/Transportation/DOF-Parking-Violation-Codes/ncbg-6agr)**  
> This dataset defines the parking violation codes in New York City and lists the fines.

### Data Platform Architecture

<img src="course_assets/project_walkthrough_images/00_02_data_platform_architecture.png" width="1000">

### Project Structure:
```bash
linkedin-learning-dq-upstream/
├── course_assets/                                     # Course materials and documentation
│   ├── product_requirements_document_template.md      # PRD template for course projects
│   ├── product_requirements_document_complete.md      # PRD completed for course projects
│   ├── project_walkthrough_complete.ipynb             # Course walkthrough
│   └── project_walkthrough_images/                    # Course walkthrough images
├── data_platform/                                     # Data platform code
│   ├── clients/                                       # Database and service clients
│   │   ├── postgres_client.py                         # PostgreSQL connection and operations
│   │   ├── minio_client.py                            # MinIO client for object storage
│   │   └── duckdb_client.py                           # DuckDB connection and operations
│   ├── config/                                        # Configuration management
│   │   ├── postgres_config.py                         # PostgreSQL connection configuration
│   │   ├── minio_config.py                            # MinIO connection configuration
│   │   └── duckdb_config.py                           # DuckDB connection configuration
│   ├── data/                                          # Data files and schemas
│   │   ├── clean_data/                                # Clean CSV data files
│   │   │   ├── dof_parking_violation_codes.csv        # Clean parking violation codes CSV file
│   │   │   └── parking_violations_issued...sample.csv # Clean parking violations issued CSV file
│   │   ├── dirty_data/                                # Dirty CSV data files
│   │   │   ├── dirty_dof_parking_violation_codes.csv  # Dirty parking violation codes CSV file
│   │   │   └── dirty_parking_violations_issued.csv    # Dirty parking violations issued CSV file
│   │   └── schemas/                                   # Data schema definitions JSON files
│   │       ├── parking_violation_codes.json           # Parking violation codes schema JSON file
│   │       └── parking_violations_issued.json         # Parking violations issued schema JSON file
│   ├── etl/                                           # ETL/ELT pipeline implementations
│   │   ├── batch_postgres_minio_etl.py                # ETL: PostgreSQL → MinIO
│   │   ├── batch_postgres_minio_wap_etl.py            # ETL: PostgreSQL → MinIO with WAP pattern
│   │   └── batch_minio_duckdb_elt.py                  # ELT: MinIO → DuckDB
│   ├── ingestion/                                     # Data ingestion modules
│   │   └── postgres_ingestion_csv.py                  # CSV file ingestion to PostgreSQL
│   ├── lakehouse/                                     # DuckDB data lakehouse
│   │   └── lakehouse.db                               # DuckDB persistent database
│   ├── scripts/                                       # Executable scripts
│   │   ├── run_ingestion.py                           # Data ingestion runner
│   │   ├── run_wap_etl.py                             # Combined ETL+ELT runner with WAP pattern
│   │   └── run_etl.py                                 # Combined ETL+ELT runner
│   ├── transactions/                                  # Transaction and concurrency modules
│   │   └── concurrency_simulator.py                   # Transaction concurrency simulation
│   └── setup_data_platform.sh                         # Data platform setup script
└── project_walkthrough.ipynb                          # Main project walkthrough
```

# Imports and Setup

In [ ]:
import logging
from data_platform.clients.postgres_client import PostgresClient
from data_platform.clients.minio_client import MinioClient
from data_platform.clients.duckdb_client import DuckdbClient

from data_platform.ingestion.postgres_ingestion_csv import PostgresIngestionCSV
from data_platform.transactions.concurrency_simulator import TransactionSimulator
from data_platform.etl.batch_postgres_minio_wap_etl import BatchPostgresMinioWriteAuditPublishETL

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s - %(name)s - %(message)s')

postgres = PostgresClient()
minio = MinioClient()

simulator = TransactionSimulator()

# Chapter 1 - Data Ingestion

Our data is already ingested from the setup of the sandbox environment. While we could look at the setup logs, it's easier to just run the data ingestion pipeline again and see the logs output.

Please note the following line that prints out logs at the debugging level within the notebook cells:

```py
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s - %(name)s - %(message)s')
```

In the below cell, let's run the following bash code and review the logs:

### 1.1 - Initial Ingestion Into PostgreSQL

In [ ]:
!python3 data_platform/scripts/run_ingestion.py

Now let's create a simple report with both ingested data assets to serve as a baseline for our data quality checks with the following query. Remember, we are less worried about the data itself and instead of the robustness of the data platform and aligning with our expectations of the data.

In [ ]:
sql_query = """
SELECT 
    parking_violation_codes.code,
    parking_violation_codes.definition,
    COUNT(parking_violations_issued.violation_code) AS ticket_count
FROM
    parking_violation_codes
JOIN
    parking_violations_issued ON
    parking_violation_codes.code = parking_violations_issued.violation_code
GROUP BY
    parking_violation_codes.code,
    parking_violation_codes.definition
ORDER BY
    ticket_count DESC
LIMIT 10;
"""

postgres.execute_query(sql_query, return_pd_dataframe=True)

### 1.2 - Ingesting Dirty Data

Now let's intentionally ingest bad data to see how the report changes and identify what improvements we can make to the data platform.

Note that this time we are using `PostgresIngestionCSV().ingest()` directly to ingest the dirty data as the ingestion scripts only ingest the clean data.

In [ ]:
PostgresIngestionCSV().ingest(
    csv_path='data_platform/data/dirty_data/dirty_parking_violations_issued.csv',
    schema_path='data_platform/data/schemas/parking_violations_issued.json'
)

PostgresIngestionCSV().ingest(
    csv_path='data_platform/data/dirty_data/dirty_dof_parking_violation_codes.csv',
    schema_path='data_platform/data/schemas/parking_violation_codes.json'
)

Then let's run the same report again and see the results.

In [ ]:
sql_query = """
SELECT 
    parking_violation_codes.code,
    parking_violation_codes.definition,
    COUNT(parking_violations_issued.violation_code) AS ticket_count
FROM
    parking_violation_codes
JOIN
    parking_violations_issued ON
    parking_violation_codes.code = parking_violations_issued.violation_code
GROUP BY
    parking_violation_codes.code,
    parking_violation_codes.definition
ORDER BY
    ticket_count DESC
LIMIT 10;
"""

postgres.execute_query(sql_query, return_pd_dataframe=True)

You should now see a row with a code of `-99` and a definition of `TEST` for 10,000 records. So despite having schema validation, we are still able to ingest bad data. In the next video, we will implement CHECK constraints to prevent this from happening.

### 1.3 - Implementing CHECK Constraints
Lets drop the tables and re-ingest the data to start fresh to see how we can place some data quality checks in place.

In [ ]:
sql_query = """
DROP TABLE IF EXISTS parking_violations_issued CASCADE;
DROP TABLE IF EXISTS parking_violation_codes CASCADE;
"""

postgres.execute_query(sql_query, return_pd_dataframe=True)

Let's run the ingestion again.

In [ ]:
!python3 data_platform/scripts/run_ingestion.py

We can confirm that the ingested data is clean again by running the same report.

In [ ]:
sql_query = """
SELECT 
    parking_violation_codes.code,
    parking_violation_codes.definition,
    COUNT(parking_violations_issued.violation_code) AS ticket_count
FROM
    parking_violation_codes
JOIN
    parking_violations_issued ON
    parking_violation_codes.code = parking_violations_issued.violation_code
GROUP BY
    parking_violation_codes.code,
    parking_violation_codes.definition
ORDER BY
    ticket_count DESC
LIMIT 10;
"""

postgres.execute_query(sql_query, return_pd_dataframe=True)

Let's now implement CHECK constraints to prevent the bad data from being ingested. We will add the following constraints to the `parking_violation_codes` table.

To learn more about CHECK constraints, see the following PostgreSQL documentation:

https://www.postgresql.org/docs/current/ddl-constraints.html

In [ ]:
sql_query = """
ALTER TABLE parking_violation_codes ADD CONSTRAINT code_check CHECK (code >= 0);
"""

postgres.execute_query(sql_query, return_pd_dataframe=True)

Let's run the ingestion again via PostgresIngestionCSV().ingest() to see the results when we try to ingest the dirty data for the `parking_violation_codes` table.

In [ ]:
PostgresIngestionCSV().ingest(
    csv_path='data_platform/data/dirty_data/dirty_dof_parking_violation_codes.csv',
    schema_path='data_platform/data/schemas/parking_violation_codes.json'
)

Amazing! We are now preventing bad data from being ingested for the `parking_violation_codes` table.

```bash
CheckViolation: new row for relation "parking_violation_codes" violates check constraint "code_check"
DETAIL:  Failing row contains (-99, TEST, null, null, 2025-09-26 22:43:18.335114, 2025-09-26 22:43:18.335114).
```

Now let's implement a similar check for the `parking_violations_issued` table.

In [ ]:
sql_query = """
ALTER TABLE parking_violations_issued
  ADD CONSTRAINT summons_number_is_ten_digits
  CHECK (summons_number::text ~ '^[0-9]{10}$');
"""

postgres.execute_query(sql_query, return_pd_dataframe=True)

Let's run the ingestion again via PostgresIngestionCSV().ingest() to see the results when we try to ingest the dirty data for the `parking_violations_issued` table.

In [ ]:
PostgresIngestionCSV().ingest(
    csv_path='data_platform/data/dirty_data/dirty_parking_violations_issued.csv',
    schema_path='data_platform/data/schemas/parking_violations_issued.json'
)

Amazing! We are now preventing bad data from being ingested for the `parking_violations_issued` table as well.

```bash
CheckViolation: new row for relation "parking_violation_codes" violates check constraint "code_check"
DETAIL:  Failing row contains (-99, TEST, null, null, 2025-09-26 22:43:18.335114, 2025-09-26 22:43:18.335114).
```

### 1.4 - Ingestion Code Exploration

Take a moment to explore the ingestion code. One thing to note is that this pipleine is using custom ingestion code, but in a more robust data platform, you would often use a "data migration tool" so that you can version control your updates to the database.

# Chapter 2 - Transactions

### 2.1 Database Transaction Pitfalls

When it comes to databases, data quality goes well beyond the schema and data itself. Specifically in production environments, we need to expect the "unexpected" and build protections against it. Now, this topic can be an entire course on its own, and I highly recommend reading *Chapter 8: Transactions* in the book *Designing Data-Intensive Applications, 2nd Edition* if you want a more in-depth look at this topic (I based this section on the book).

With that said, for a free resource that the above book references, I again recommend PostgresSQL's official documentaton:
- [PostgreSQL Documentation - Chapter 13. Concurrency Control](https://www.postgresql.org/docs/current/mvcc.html)
- [PostgreSQL Documentation - 13.2. Transaction Isolation](https://www.postgresql.org/docs/current/transaction-iso.html)
- [PostgreSQL Documentation - SET TRANSACTION](https://www.postgresql.org/docs/current/sql-set-transaction.html)

A situation that we will focus on in this chapter is the "lost update" problem, which is a specific manifestation of a serialization anomaly. This occurs when two transactions read the same data, make changes, and then one of the changes is lost, resulting in an outcome that would be impossible if the transactions were executed one at a time in any order.

The following diagram shows the problem:

<img src="course_assets/project_walkthrough_images/02_01_concurrency.png" width="1000">

This is super theory heavy, furthermore, it's hard to replicate this organically as we are not working with a multi-user environment. So instead, we will simulate the problem using the `TransactionSimulator` class that we initiated in the setup of the sandbox environment.

### 2.2 - Simulating the Lost Update Problem

We will first simulate the problem without any defined isolation levels. Given that the failures are random, we are likely to get different results each time we run the simulation (this is expected and the exact problem we are trying to highlight).

> [!NOTE]
> By default, PostgreSQL uses the `READ COMMITTED` isolation level in the background of all queries.

In [ ]:
simulator.simulate(increment_a=1, increment_b=2, iterations=5)

Now let's run the simulation again with the same settings and see how the results change. You may need to re-run the cell due to the small chance that the two random results are the same.

In [ ]:
simulator.simulate(increment_a=1, increment_b=2, iterations=5)

### 2.3 - Preventing Concurrency Issues With Isolation

You can now see that the same set of transactions are resulting in different numbers! This is how the database systems themselves result in data quality issues, and why we need to be proactive about it.

Now let's see how the problem is solved by setting the isolation level to `SERIALIZABLE`.

In [ ]:
simulator.simulate(increment_a=1, increment_b=2, iterations=5, isolation_level="SERIALIZABLE")

Our result is now the expected value of 15! But let's run it again to confirm that the result is always the same.

In [ ]:
simulator.simulate(increment_a=1, increment_b=2, iterations=5, isolation_level="SERIALIZABLE")

They are the same! I want you to pay special attention to the logs of the simulation. You can see that the transactions are now being executed in order and the result is always the same.

Specifically check out the logs:

```bash
WARNING - data_platform.transactions.concurrency_simulator - Transaction failed: could not serialize access due to concurrent update
INFO - data_platform.transactions.concurrency_simulator - Retry attempt 1/4
```

We can see that the transactions are now being executed in order and will retry a failed transaction up to 4 times before giving up.

### 2.4 - Ingestion Code Exploration

Take a moment to explore the transactions code. One thing to note is that this is only one type of concurrency problem that can occur, but this approach of isolation at the serializable level is the most conservative. The tradeoff is that it's much slower, so you need to be mindful of the performance need of your business use case.

# Chapter 3 - Replication

### 3.1 - A Brief Overview of Data Lakehouses

<img src="course_assets/project_walkthrough_images/00_02_data_platform_architecture.png" width="1000">

Let's first clean up the PostgreSQL database to start fresh.

In [ ]:
sql_query = """
DROP TABLE IF EXISTS parking_violations_issued CASCADE;
DROP TABLE IF EXISTS parking_violation_codes CASCADE;
DROP TABLE IF EXISTS transaction_simulation CASCADE;
"""
postgres.execute_query(sql_query, return_pd_dataframe=True)

In [ ]:
query = """
SELECT
    table_name
FROM
    information_schema.tables
WHERE
    table_schema = 'public'
ORDER BY
    table_name;
"""
postgres.execute_query(query, return_pd_dataframe=True)

We can now run the ingestion and etl pipelines again.

In [ ]:
!python3 data_platform/scripts/run_ingestion.py

In [ ]:
!python3 data_platform/scripts/run_etl.py

To confirm that the data files are in the MinIO bucket, we can run the following cell code:

In [ ]:
objects = minio.list_objects("raw-data")
print("Files in raw-data bucket:")
for obj in sorted(objects):
    print(f"  {obj}")

Pay special attention to how we have multiple files for the same data assets! This allows us to have a snapshot of the data at the time of replication. When DuckDB reads the data, it will read the latest file for each data asset.

Now let's run the same report, but in DuckDB instead of PostgreSQL, as before to confirm that the data is consistent.


In [ ]:
query = """
SELECT 
    parking_violation_codes.code,
    parking_violation_codes.definition,
    COUNT(parking_violations_issued.violation_code) AS ticket_count
FROM
    staging.parking_violation_codes
JOIN
    staging.parking_violations_issued ON
    parking_violation_codes.code = parking_violations_issued.violation_code
GROUP BY
    parking_violation_codes.code,
    parking_violation_codes.definition
ORDER BY
    ticket_count DESC
LIMIT 10;
"""
with DuckdbClient(db_path="data_platform/lakehouse/lakehouse.db") as duckdb:
    display(duckdb.execute_query(query, return_pd_dataframe=True))

Confirmed that the data is consistent!

### 3.2 - Intentionally Breaking Our Replication Pipeline

Now let's break the pipeline by adding dirty data into the data platform. Note that our previous ingestion checks in PostgreSQL are no longer present since we dropped the tables to start fresh. This is intentional so we can persist the dirty data to this stage of the pipeline.

Since it's dirty data, we will need to run the manual ingestion process again.

In [ ]:
PostgresIngestionCSV().ingest(
    csv_path='data_platform/data/dirty_data/dirty_parking_violations_issued.csv',
    schema_path='data_platform/data/schemas/parking_violations_issued.json'
)

PostgresIngestionCSV().ingest(
    csv_path='data_platform/data/dirty_data/dirty_dof_parking_violation_codes.csv',
    schema_path='data_platform/data/schemas/parking_violation_codes.json'
)

Now let's run the same report again and see the results and confirm the dirty data is present.

In [ ]:
sql_query = """
SELECT 
    parking_violation_codes.code,
    parking_violation_codes.definition,
    COUNT(parking_violations_issued.violation_code) AS ticket_count
FROM
    parking_violation_codes
JOIN
    parking_violations_issued ON
    parking_violation_codes.code = parking_violations_issued.violation_code
GROUP BY
    parking_violation_codes.code,
    parking_violation_codes.definition
ORDER BY
    ticket_count DESC
LIMIT 10;
"""

postgres.execute_query(sql_query, return_pd_dataframe=True)

Let's now run the ETL pipeline again to see the results.

In [ ]:
!python3 data_platform/scripts/run_etl.py

It looks like the dirty data is now causing the transformation to Avro files to fail. This has huge implications on data timeliness of the data lakehouse for downstream data consumers.

### 3.3 - Introduction to the Write Audit Publish (WAP) Pattern

We need to add a data quality check on the pipeline that does the following:
1. Ensures that the data pipeline doesn't fail and not fully replicate the data.
2. Flag data that will cause errors.
3. Quarantine the data for manual review later.

Thus, we need to implement the Write Audit Publish (WAP) Pattern on our replication pipeline as illustrated in the following diagram:

<img src="course_assets/project_walkthrough_images/03_01_wap_batch.png" width="1000">

### 3.4 - Implementing the Write Audit Publish Pattern

Let's first confirm that our data is present in PostgreSQL.

In [ ]:
postgres = PostgresClient()

query = """
SELECT
    table_name
FROM
    information_schema.tables
WHERE
    table_schema = 'public'
ORDER BY
    table_name;
"""
postgres.execute_query(query, return_pd_dataframe=True)

Now let's initialize the WAP class and run the WAP ETL pipeline.

In [ ]:
write_audit_publish = BatchPostgresMinioWriteAuditPublishETL(
    bucket_name="raw-data",
    schema_directory="data_platform/data/schemas"
)
write_audit_publish.run_wap()
write_audit_publish.run_etl()

Success! Now let's confirm that the data is present in PostgreSQL.

In [ ]:
from data_platform.clients.postgres_client import PostgresClient
postgres = PostgresClient()

query = """
SELECT
    table_name
FROM
    information_schema.tables
WHERE
    table_schema = 'public'
ORDER BY
    table_name;
"""
postgres.execute_query(query, return_pd_dataframe=True)

We can see that the data is present in PostgreSQL, specifically the quarantine and staging tables.

Now let's take a look at the data in the staging and quarantine tables to ensure that the data is present and the data quality checks are working as expected.

In [ ]:
sql_query = """
select * from staging_parking_violation_codes limit 10;
"""
postgres.execute_query(sql_query, return_pd_dataframe=True)

In [ ]:
sql_query = """
select * from quarantine_parking_violation_codes limit 10;
"""
postgres.execute_query(sql_query, return_pd_dataframe=True)

In [ ]:
sql_query = """
select * from staging_parking_violations_issued limit 10;
"""
postgres.execute_query(sql_query, return_pd_dataframe=True)

In [ ]:
sql_query = """
select * from quarantine_parking_violations_issued limit 10;
"""
postgres.execute_query(sql_query, return_pd_dataframe=True)

Let's now confirm that the staging data is present in the MinIO bucket too.

In [ ]:
objects = minio.list_objects("raw-data")
print("Files in raw-data bucket:")
for obj in sorted(objects):
    print(f"  {obj}")

Now let's run the full script end-to-end to see the results and logs!

In [ ]:
!python3 data_platform/scripts/run_wap_etl.py

Success! The logs confirm the pipeline works, and we see both the original raw data in DuckDB and the new staging data from the WAP pipeline.

Finally, let's confirm that the raw data and the staging data result in the same report.

In [ ]:
query = """
SELECT 
    parking_violation_codes.code,
    parking_violation_codes.definition,
    COUNT(parking_violations_issued.violation_code) AS ticket_count
FROM
    staging.parking_violation_codes
JOIN
    staging.parking_violations_issued ON
    parking_violation_codes.code = parking_violations_issued.violation_code
GROUP BY
    parking_violation_codes.code,
    parking_violation_codes.definition
ORDER BY
    ticket_count DESC
LIMIT 10;
"""
with DuckdbClient(db_path="data_platform/lakehouse/lakehouse.db") as duckdb:
    display(duckdb.execute_query(query, return_pd_dataframe=True))

In [ ]:
query = """
SELECT 
    staging_parking_violation_codes.code,
    staging_parking_violation_codes.definition,
    COUNT(staging_parking_violations_issued.violation_code) AS ticket_count
FROM
    staging.staging_parking_violation_codes
JOIN
    staging.staging_parking_violations_issued ON
    staging_parking_violation_codes.code = staging_parking_violations_issued.violation_code
GROUP BY
    staging_parking_violation_codes.code,
    staging_parking_violation_codes.definition
ORDER BY
    ticket_count DESC
LIMIT 10;
"""
with DuckdbClient(db_path="data_platform/lakehouse/lakehouse.db") as duckdb:
    display(duckdb.execute_query(query, return_pd_dataframe=True))

Success!

### 3.5 - Replication Code Exploration

Take a moment to explore the replication code. One thing to note is that the Write Audit Publish pattern can also be used for streaming data where it's often the most popular (but outside the scope of this course).

# Chapter 4 - Data Platform Product Requirements Document (PRD)

We now have all of the information we need to create the Product Requirements Document (PRD).

Below is the PRD outline to fill out for the data platform. If you want an extra challenge, take a moment to try and fill it out yourself before watching the video where we can compare what we wrote.

# Data Platform Product Requirements Document (PRD)

## 1. Data Platform Architecture Overview

### Current Architecture

![](course_assets/project_walkthrough_images/00_02_data_platform_architecture.png)

[Describe the existing data platform in 1-2 paragraphs. Include a simple diagram showing data flow from sources to analytics.]

**Key Components:**
- Data Sources: [List main data sources]
- Ingestion: [How data gets into the system]
- Database: [PostgreSQL setup]
- Data Lake: [MinIO storage]
- ETL Pipelines: [Data processing]

## 2. Gap Analysis

### 2.1 Ingestion into Transactional Database

Here's the current state in markdown format:

**Current State:**

- [Brief description]

**Gaps Found:**
- [Gap 1]
- [Gap 2]
**Recommendations:**
- [Recommendation 1]
- [Recommendation 2]

### 2.2 Transactions on Transactional Database
**Current State:**

- [Brief description]

**Gaps Found:**
- [Gap 1]
- [Gap 2]
**Recommendations:**
- [Recommendation 1]
- [Recommendation 2]

### 2.3 Replication to Data Lakehouse
**Current State:**

- [Brief description]
**Gaps Found:**
- [Gap 1]
- [Gap 2]
**Recommendations:**
- [Recommendation 1]
- [Recommendation 2]

## 3. Improvement Suggestions

### Priority 1 (High Impact)
**Improvement:** [Title]
- What: [Brief description]
- Why: [Benefits]
- Tradeoffs: [Pros and cons]
- Test Results: [What you found when testing]

### Priority 2 (Medium Impact)
**Improvement:** [Title]
- What: [Brief description]
- Why: [Benefits]
- Tradeoffs: [Pros and cons]
- Test Results: [What you found when testing]

### Priority 3 (Low Impact)
**Improvement:** [Title]
- What: [Brief description]
- Why: [Benefits]
- Tradeoffs: [Pros and cons]
- Test Results: [What you found when testing]

## 4. Remaining Questions

**Questions outside this project scope:**
- [Question 1]: [Why it's important but not covered]
- [Question 2]: [Why it's important but not covered]
- [Question 3]: [Why it's important but not covered]

## 5. Next Steps

**Immediate Actions:**
- [Action 1]
- [Action 2]

**Future Considerations:**
- [Consideration 1]
- [Consideration 2]

# END OF PROJECT WALKTHROUGH